In [3]:
%pip -q install torchgeo scikit-learn

In [4]:
from google.colab import drive
drive.mount('/content/drive') # for saving model
# colab is slow for small files in a mounted drive, this copies it to a local colab disk
!cp "/content/drive/My Drive/Dissertation/Datasets/ETCI2021_filtered_40pct.zip" "/content/ETCI2021_filtered_40pct.zip"

# Unzip it locally (quietly)
!unzip -q -o "/content/ETCI2021_filtered_40pct.zip" -d "/content/ETCI2021_filtered_40pct"

Mounted at /content/drive


In [5]:
import math
import cv2
import matplotlib.pyplot as plt
import os
import numpy as np
from pathlib import Path
from glob import glob
from tqdm import tqdm
import torch
import tensorflow as tf
import h5py

folder_models = ('/content/drive/My Drive/Dissertation/models/')
DS_ROOT = Path('/content/ETCI2021_filtered_40pct/ETCI2021_filtered_40pct/data/') #unzipped so nested folder
if not DS_ROOT.exists():
  raise FileNotFoundError("Couldn't find copied dataset")

# Referenced from: https://medium.com/cloud-to-street/jumpstart-your-machine-learning-satellite-competition-submission-2443b40d0a5a

**Setup dataframes**

In [6]:
def concatenate_SAR(vv_path, vh_path):
    vv_img = cv2.imread(vv_path, cv2.IMREAD_GRAYSCALE)
    vh_img = cv2.imread(vh_path, cv2.IMREAD_GRAYSCALE)
    # ratio = vv_img - vh_img
    if vv_img is  None:
        raise FileExistsError(f"Could not open vv file with path {vv_path}")
    if vh_img is None:
        raise FileExistsError(f"Could not open vh file with path {vh_path}")

    concatenated_SAR = np.stack((vv_img, vh_img), axis=-1)
    if concatenated_SAR.shape != (*vv_img.shape, 2):
        raise ValueError("Final concatenated image shape mismatch.")
    return concatenated_SAR

def append_ratio(dataset):
    """
    Add a third channel representing VV - VH.

    Expected input:
        (N, 256, 256, 2)

    Returns:
        (N, 256, 256, 3)
    """

    dataset = np.asarray(dataset, dtype=np.float32)

    # Check input dimensions
    if dataset.ndim != 4:
        raise ValueError(
            f"Expected dataset with 4 dimensions (N, H, W, C), "
            f"but got shape {dataset.shape}"
        )

    if dataset.shape[-1] != 2:
        raise ValueError(
            f"Expected 2 input channels (VV, VH), "
            f"but got {dataset.shape[-1]}"
        )

    # Extract VV and VH
    vv = dataset[..., 0]
    vh = dataset[..., 1]

    # Calculate ratio/difference channel
    ratio = vv - vh

    # Add channel dimension back
    ratio = ratio[..., np.newaxis]

    # Concatenate: VV, VH, ratio
    dataset_with_ratio = np.concatenate(
        [dataset, ratio],
        axis=-1
    )

    # Final validation
    if dataset_with_ratio.shape[-1] != 3:
        raise ValueError(
            f"Expected 3 channels after adding ratio, "
            f"but got shape {dataset_with_ratio.shape}"
        )

    return np.ascontiguousarray(dataset_with_ratio, dtype=np.float32)

def find_vv(flood_label_path: Path) -> Path:
    file_name = flood_label_path.name
    vv_path = flood_label_path.parent.parent / "vv" / file_name
    vv_path = vv_path.with_name(f"{vv_path.stem}_vv{vv_path.suffix}")
    if vv_path.exists():
        return vv_path
    else:
        print(vv_path)
        raise FileNotFoundError(f"Couldn't find vv file for {flood_label_path}")

def find_vh(flood_label_path: Path) -> Path:
    file_name = flood_label_path.name
    vh_path = flood_label_path.parent.parent / "vh" / file_name
    vh_path = vh_path.with_name(f"{vh_path.stem}_vh{vh_path.suffix}")
    if vh_path.exists():
        return vh_path
    else:
        print(vh_path)
        raise FileNotFoundError(f"Couldn't find vh file for {flood_label_path}")

def fetch_train_test_data_from_split(ds_root, split):
    X = []
    Y = []
    if split not in ["train", "test"]:
      raise ValueError("Split needs to be train or test")
    split_path = ds_root / split
    if not split_path.exists():
        return None, None

    all_flood_labels = list(split_path.glob("*/tiles/flood_label/*.png"))
    print(f"Processing {len(all_flood_labels)} images for {split}ing split...")

    for flood_label in tqdm(all_flood_labels):
        try:
            vv_path = find_vv(flood_label)
            vh_path = find_vh(flood_label)
            fl_img = cv2.imread(str(flood_label), cv2.IMREAD_GRAYSCALE)
            SAR_img = concatenate_SAR(vv_path, vh_path)
            X.append(SAR_img)
            Y.append(fl_img)

        except (FileNotFoundError, FileExistsError) as e:
            continue #maybe something else
    return X, Y


LOAD_PREPROCESSED_DATA = True

if not LOAD_PREPROCESSED_DATA:
  X_Train, Y_Train = fetch_train_test_data_from_split(DS_ROOT, "train")
  if X_Train is None or Y_Train is None:
    print("Arrays have not been populated, likely due to a path error")
  # X: (256, 256, 3)
  # Y: (256, 256, 1)
else:
  with h5py.File('/content/drive/My Drive/Dissertation/Datasets/ETCI2021_40_pct_preprocessed_RFI.h5', 'r') as f:
    X_Train = f['X_Train'][:]
    Y_Train = f['Y_Train'][:]
    X_Train_min = f.attrs["X_Train_min"]
    X_Train_max = f.attrs["X_Train_max"]
    f.close()

Y_Train = (Y_Train / 255.0).astype(np.float32)

# need to work out class weights
# Check labels
print("Unique Y values:", np.unique(Y_Train))

# Calculate class balance
flood_pixels = np.sum(Y_Train == 1.)
total_pixels = Y_Train.size

pct_flood = flood_pixels / total_pixels

FOCAL_ALPHA = round(1 - pct_flood, 3)
FLOOD_WEIGHT = round((1 - pct_flood) * 10.0, 3)

print(f"Flood pixels: {pct_flood:.3%}")
print(f"Focal alpha: {FOCAL_ALPHA}")
print(f"Flood weight: {FLOOD_WEIGHT}")


Unique Y values: [0. 1.]
Flood pixels: 59.663%
Focal alpha: 0.403
Flood weight: 4.034


In [7]:
# X_Train = X_Train.numpy() # they are already np
# Y_Train = Y_Train.numpy() # they are already np

X_augmented = np.concatenate([
    X_Train,
    np.flip(X_Train, axis=1),
    np.flip(X_Train, axis=2),
    np.rot90(X_Train, axes=(1, 2))
], axis=0)

Y_augmented = np.concatenate([
    Y_Train,
    np.flip(Y_Train, axis=1),
    np.flip(Y_Train, axis=2),
    np.rot90(Y_Train, axes=(1,2))
], axis=0)

X_Train_processed = np.ascontiguousarray(
    X_augmented,
    dtype=np.float32
)

Y_Train_processed = np.ascontiguousarray(
    Y_augmented,
    dtype=np.float32
)

X_Train_processed = append_ratio(X_Train_processed)
N = X_Train_processed.shape[0]
#10% for val
X_Val_processed = X_Train_processed[-int(0.9*N):]
Y_Val_processed = Y_Train_processed[-int(0.9*N):]
X_Train_processed = X_Train_processed[:int(0.9*N)]
Y_Train_processed = Y_Train_processed[:int(0.9*N)]

# permuting
X_Train_processed = np.transpose(X_Train_processed, (0, 3, 1, 2))
X_Val_processed = np.transpose(X_Val_processed, (0, 3, 1, 2))
print("Xt:", X_Train_processed.shape)
print("Xv:", X_Val_processed.shape)
print("Yt:", Y_Train_processed.shape)
print("Yv:", Y_Val_processed.shape)

Xt: (1062, 3, 256, 256)
Xv: (1062, 3, 256, 256)
Yt: (1062, 256, 256)
Yv: (1062, 256, 256)


# Data preprocessing

# Model

In [ ]:
from torchgeo.models import unet, Unet_Weights
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torch.utils.data import TensorDataset, DataLoader
DEVICE = "cuda" if torch.cuda.is_available() else 'cpu'
LOAD_MODEL = False
model_name = "Floodmapping_UNet_RFI.pt"
BATCH_SIZE = 64

train_dataset = TensorDataset(
    torch.tensor(X_Train_processed, dtype=torch.float32),
    torch.tensor(Y_Train_processed, dtype=torch.float32)
)

val_dataset = TensorDataset(
    torch.tensor(X_Val_processed, dtype=torch.float32),
    torch.tensor(Y_Val_processed, dtype=torch.float32)
)
# train_dataset = TensorDataset(torch.stack(X_Train), torch.tensor(Y_Train, dtype=torch.float32))
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
# val_dataset = TensorDataset(torch.stack(X_Val), torch.tensor(Y_Val, dtype=torch.float32))
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=True)

def batch_iou(logits, masks, threshold=0.5, eps=1e-7):
    """
    logits: [B, 1, H, W]
    masks:  [B, 1, H, W]
    """

    probs = torch.sigmoid(logits)
    preds = (probs > threshold).float()

    masks = masks.float()

    # Intersection and union per batch
    intersection = (preds * masks).sum()
    union = preds.sum() + masks.sum() - intersection

    iou = (intersection + eps) / (union + eps)

    return iou

if not LOAD_MODEL:
  model = unet(in_chans=3, classes=1)
  model = model.to(DEVICE)
  optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)


  EPOCHS = 100
  MIN_EPOCHS = 10
  PATIENCE = 10
  best_val_loss = float("inf")
  epochs_without_improvement = 0
  train_loss_per_epoch = []
  val_loss_per_epoch = []
  train_IoU_per_epoch = []
  val_IoU_per_epoch = []

  for epoch in range(EPOCHS):
    model.train()
    running_train_iou = 0.0
    running_val_iou = 0.0
    running_train_loss = 0.0
    running_val_loss = 0.0

    for x_train, y_train in train_loader:
      x_train = x_train.to(DEVICE)
      y_train = y_train.to(DEVICE).unsqueeze(1)

      optimizer.zero_grad()
      logits = model(x_train)

      loss = torchvision.ops.sigmoid_focal_loss(
          inputs=logits, # not sure it is logits
          targets=y_train,
          alpha=1-FOCAL_ALPHA,
          gamma=2.0,
          reduction="mean"
      )

      running_train_loss += loss.item()
      train_loss_per_epoch.append(loss.item())

      loss.backward()
      optimizer.step()

    train_iou = batch_iou(logits, y_train).item()
    train_IoU_per_epoch.append(train_iou)
    running_train_loss /= len(train_loader)

    model.eval()
    with torch.no_grad():
      for x_val, y_val in val_loader:
        x_val = x_val.to(DEVICE)
        y_val = y_val.to(DEVICE).unsqueeze(1)

        logits = model(x_val)

        loss = torchvision.ops.sigmoid_focal_loss(
            inputs=logits, # not sure it is logits
            targets=y_val,
            alpha=FOCAL_ALPHA,
            gamma=2.0,
            reduction="mean"
        )

        running_val_loss += loss.item()
        val_loss_per_epoch.append(loss.item())
      val_iou = batch_iou(logits, y_val).item()
      val_IoU_per_epoch.append(val_iou)

    running_val_loss /= len(val_loader)

    print(
    f"Epoch [{epoch + 1}/{EPOCHS}] | "
    f"Train Loss: {running_train_loss:.4f} | Train IoU: {train_iou:.4f} | "
    f"Val Loss: {running_val_loss:.4f} | Val IoU: {val_iou:.4f}"
    )

    # Save best model and early stopping check
    if running_val_loss < best_val_loss and epoch > MIN_EPOCHS: # track val loss
      best_val_loss = running_val_loss
      epochs_without_improvement = 0
      torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_loss": running_train_loss,
            "val_loss": running_val_loss,
        }, folder_models + model_name
      )
      print("-- Best model saved")
    else:
      epochs_without_improvement += 1
      if epochs_without_improvement >= PATIENCE and epoch > MIN_EPOCHS:
          print(
              f"\nEarly stopping at epoch {epoch + 1}. "
              f"Best validation MAE: {best_val_loss:.4f}"
          )
          break

else:
  checkpoint = torch.load(folder_models + model_name, map_location="cpu")
  model = unet(in_chans=3)
  model.load_state_dict(checkpoint["model_state_dict"])

model.eval() # set to inference mode

Epoch [1/100] | Train Loss: 0.1048 | Train IoU: 0.7044 | Val Loss: 0.0782 | Val IoU: 0.3493
Epoch [2/100] | Train Loss: 0.0526 | Train IoU: 0.7811 | Val Loss: 0.0809 | Val IoU: 0.5858
Epoch [3/100] | Train Loss: 0.0406 | Train IoU: 0.8273 | Val Loss: 0.0526 | Val IoU: 0.7396
Epoch [4/100] | Train Loss: 0.0362 | Train IoU: 0.8198 | Val Loss: 0.0368 | Val IoU: 0.8002
Epoch [5/100] | Train Loss: 0.0335 | Train IoU: 0.8655 | Val Loss: 0.0336 | Val IoU: 0.7957
Epoch [6/100] | Train Loss: 0.0328 | Train IoU: 0.7849 | Val Loss: 0.0332 | Val IoU: 0.8400
Epoch [7/100] | Train Loss: 0.0307 | Train IoU: 0.8477 | Val Loss: 0.0306 | Val IoU: 0.8657
Epoch [8/100] | Train Loss: 0.0286 | Train IoU: 0.8322 | Val Loss: 0.0296 | Val IoU: 0.8920
Epoch [9/100] | Train Loss: 0.0274 | Train IoU: 0.8074 | Val Loss: 0.0300 | Val IoU: 0.8598
Epoch [10/100] | Train Loss: 0.0266 | Train IoU: 0.8813 | Val Loss: 0.0271 | Val IoU: 0.8645
Epoch [11/100] | Train Loss: 0.0249 | Train IoU: 0.8773 | Val Loss: 0.0252 | Va

In [ ]:
total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

param_size = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
)

buffer_size = sum(
    b.numel() * b.element_size()
    for b in model.buffers()
)

model_size_mb = (param_size + buffer_size) / (1024 ** 2)

print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Model size:           {model_size_mb:.2f} MB")

In [ ]:
flood_iou = train_IoU_per_epoch
val_flood_iou = val_IoU_per_epoch
plt.figure(figsize=(6,6))
plt.plot(np.linspace(0, len(flood_iou), len(flood_iou)), flood_iou)
plt.plot(np.linspace(0, len(val_flood_iou), len(val_flood_iou)), val_flood_iou)
plt.yticks(np.arange(0, 1.01, 0.05))
plt.ylim([0.5, 1])
plt.grid(True)
plt.xlabel("Epochs")
plt.ylabel("Intersecion over union (IoU)")
plt.title("IoU per epoch during U-Net training with RFI")
plt.legend(["Training IoU", "Validation IoU"])
plt.savefig("/content/drive/My Drive/Dissertation/figures/unet_iou_training_RFI.png")
plt.tight_layout()
plt.show()

# Testing model

In [ ]:
# augment
with h5py.File("/content/drive/My Drive/Dissertation/Datasets/ETCI2021_40_pct_preprocessed_Test_RFI.h5", 'r') as f:
  X_Test = f['X_Test'][:]
  Y_Test = f['Y_Test'][:]
  f.close()

# X_Train = X_Train.numpy() # they are already np
# Y_Train = Y_Train.numpy() # they are already np

X_Test_augmented = np.concatenate([
    X_Test,
    np.flip(X_Test, axis=1),
    np.flip(X_Test, axis=2),
    # np.rot90(X_Test, axes=(1,2))
], axis=0)

Y_Test_augmented = np.concatenate([
    Y_Test,
    np.flip(Y_Test, axis=1),
    np.flip(Y_Test, axis=2),
    # np.rot90(Y_Test, axes=(1,2))
], axis=0)

X_Test_processed = np.ascontiguousarray(
    X_Test_augmented,
    dtype=np.float32
)

Y_Test_processed = np.ascontiguousarray(
    Y_Test_augmented,
    dtype=np.float32
)

X_Test_processed = append_ratio(X_Test_processed)
X_Test_processed = torch.tensor(X_Test_processed, dtype=torch.float32)
X_Test_processed = X_Test_processed.permute(0, 3, 1, 2)
X_Test_processed = X_Test_processed.to(DEVICE)

print("X:", X_Test_processed.shape)
print("Y:", Y_Test_processed.shape)


In [ ]:
# inference test
import time
sample = X_Test_processed[5]
# sample = sample.permute(2, 0, 1)
# sample = sample.to(DEVICE)
sample = sample.unsqueeze(0)
all_times = []
for _ in range(10):
  start = time.time()
  _ = model(sample)
  all_times.append(time.time() - start)
print(f"Avg inference: {round(np.mean(all_times) * 1e3, 3)} ms")

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, jaccard_score
torch.cuda.empty_cache()

with torch.no_grad():
  predictions = model(X_Test_processed)
  # visualise some
  for i in range(5):
    pred = predictions[i]
    true = Y_Test_processed[i]
    pred = (pred >= 0.5).int().squeeze().cpu().numpy()
    true = (true >= 0.5).astype(int)
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))

    # Plot Ground Truth
    axes[0].imshow(true, cmap="gray")
    axes[0].set_title(f"True Label {i+1}")
    axes[0].axis("off")

    # Plot Prediction
    axes[1].imshow(pred, cmap="gray")
    axes[1].set_title(f"Predicted {i+1}")
    axes[1].axis("off")

    # Display this specific figure
    plt.tight_layout()
    path = f"/content/drive/My Drive/Dissertation/figures/sample_{str(i)}_unet_RFI.png"
    plt.savefig(path)
    plt.show()

  def dice_score(y_true, y_pred, threshold=0.5):
      y_true = np.asarray(y_true).astype(bool)
      y_pred = (np.asarray(y_pred) >= threshold)

      intersection = np.sum(y_true & y_pred)

      return (2.0 * intersection) / (
          np.sum(y_true) + np.sum(y_pred) + 1e-7
      )


  # Metric evaluation
  y_pred = (predictions >= 0.5).int().squeeze().cpu().numpy()
  y_true = (Y_Test_processed >= 0.5).astype(int)

  # Lists to store metrics for each image
  img_ious = []
  img_f1s = []
  img_precisions = []
  img_recalls = []
  img_dices = []

  # Loop through every single image in the test set
  for i in range(len(X_Test_processed)):
      # Flatten just the current single image's masks
      true_img = y_true[i].flatten()
      pred_img = y_pred[i].flatten()

      # Check if there are ANY actual flood pixels or predicted flood pixels
      # This avoids 0/0 division anomalies for completely dry images
      if np.sum(true_img) == 0 and np.sum(pred_img) == 0:
          # Perfect prediction of no flood!
          iou = 1.0
          f1 = 1.0
          precision = 1.0
          recall = 1.0
      else:
          # Calculate standard binary metrics for the flood class (1)
          # zero_division=0 ensures that if it predicts no flood, it doesn't crash
          iou = jaccard_score(true_img, pred_img, pos_label=1, zero_division=0)
          f1 = f1_score(true_img, pred_img, pos_label=1, zero_division=0)
          precision = precision_score(true_img, pred_img, pos_label=1, zero_division=0)
          recall = recall_score(true_img, pred_img, pos_label=1, zero_division=0)
          dice = dice_score(true_img, pred_img)

      img_ious.append(iou)
      img_f1s.append(f1)
      img_precisions.append(precision)
      img_recalls.append(recall)
      img_dices.append(dice)

# Calculate the mean across the whole test set
print(f"Mean Per-Image IoU:         {np.mean(img_ious):.4f}")
print(f"Mean Per-Image F1 Score:    {np.mean(img_f1s):.4f}")
print(f"Mean Per-Image Precision:   {np.mean(img_precisions):.4f}")
print(f"Mean Per-Image Recall:      {np.mean(img_recalls):.4f}")
print(f"Mean Per-Image Dice Score:  {np.mean(img_dices):.4f}")

predictions = predictions.cpu().numpy()
X_Test_processed = X_Test_processed.cpu().numpy()
print("Saving results")
with h5py.File("/content/drive/My Drive/Dissertation/data/UNet_results_RFI.h5", 'w') as f:
  f.create_dataset("y_pred", data=predictions)
  f.create_dataset("y_true", data=X_Test_processed)
  f.attrs["iou"] = float(np.mean(img_ious))
  f.attrs["f1"] = float(np.mean(img_f1s))
  f.attrs["precision"] = float(np.mean(img_precisions))
  f.attrs["recall"] = float(np.mean(img_recalls))
  f.attrs["dice"] = float(np.mean(img_dices))
  f.close()
print("Saved.")